In [1]:
import pandas as pd
import numpy as np
from fredapi import Fred
import yfinance as yf
import requests
from datetime import datetime

print("All imports successful")
print(f"Project: GH-Yield")
print(f"Date: {datetime.now().strftime('%Y-%m-%d')}")

All imports successful
Project: GH-Yield
Date: 2026-06-20


In [8]:

fred = Fred(api_key='c4993cfe80c26bde2cfe1de7991ca32d')  # Replace with your actual key

# US T-bill
us_tbill = fred.get_series('DTB3', observation_start='2015-01-01')
us_tbill = us_tbill.reset_index()
us_tbill.columns = ['date', 'us_tbill_3m']
us_tbill.to_csv('../data/us_tbill_3m.csv', index=False)
print(f"US T-bill: {len(us_tbill)} rows saved")

# Fed Funds
fed_funds = fred.get_series('FEDFUNDS', observation_start='2015-01-01')
fed_funds = fed_funds.reset_index()
fed_funds.columns = ['date', 'fed_funds']
fed_funds.to_csv('../data/fed_funds_rate.csv', index=False)
print(f"Fed Funds: {len(fed_funds)} rows saved")

# USD/GHS
data = yf.download('USDGHS=X', start='2015-01-01', progress=False)
fx_clean = data['Close'].reset_index()
fx_clean.columns = ['date', 'usd_ghs']
fx_clean.to_csv('../data/usd_ghs.csv', index=False)
print(f"USD/GHS: {len(fx_clean)} rows saved")

print("\nAll data saved!")

US T-bill: 2990 rows saved
Fed Funds: 137 rows saved
USD/GHS: 2984 rows saved

All data saved!


In [19]:
import wbgapi as wb

# Ghana Inflation
inflation = wb.data.DataFrame('FP.CPI.TOTL.ZG', economy='GHA', time=range(2010, 2026))
inflation.to_csv('../data/ghana_inflation.csv')
print(f"Inflation: {inflation.shape} saved")

# Ghana GDP
gdp = wb.data.DataFrame('NY.GDP.MKTP.CD', economy='GHA', time=range(2010, 2026))
gdp.to_csv('../data/ghana_gdp.csv')
print(f"GDP: {gdp.shape} saved")

print("\nAll data saved!")

Inflation: (1, 16) saved
GDP: (1, 16) saved

All data saved!


In [9]:
import requests
import warnings
warnings.filterwarnings('ignore')

# Download BoG Summary of Economic and Financial Data PDF
url = 'https://www.bog.gov.gh/wp-content/uploads/2026/01/Summary-of-Economic-and-Financial-Data-January-2026.pdf'
response = requests.get(url, verify=False)

print(f"Status: {response.status_code}")
print(f"Size: {len(response.content)} bytes")

if response.status_code == 200:
    with open('../data/bog_summary_jan2026.pdf', 'wb') as f:
        f.write(response.content)
    print("PDF saved!")
else:
    print("Download failed")

Status: 200
Size: 807977 bytes
PDF saved!


In [10]:
import pdfplumber

with pdfplumber.open('../data/bog_summary_jan2026.pdf') as pdf:
    print(f"Total pages: {len(pdf.pages)}")
    for i, page in enumerate(pdf.pages[:6]):
        text = page.extract_text()
        if text and '91' in text:
            print(f"\n=== Page {i} ===")
            print(text[:3000])
            break

Total pages: 16

=== Page 2 ===
27-Jan-2026 18:02:56 Summary of Macroeconomic and Financial Data c1
1. Price Developments
2024:12 2025:01 2025:02 2025:03 2025:04 2025:05 2025:06 2025:07 2025:08 2025:09 2025:10 2025:11 2025:12
Consumer Price Index i
All Consumer Prices 248.3 252.6 255.9 256.5 258.6 260.5 257.3 259.1 255.7 258.0 257.0 259.4 261.7
... Food Prices 277.5 283.2 288.2 287.7 290.2 292.8 291.4 293.2 285.9 287.5 284.9 287.9 291.0
... Non-Food Prices 226.0 229.2 231.2 232.7 234.3 235.7 231.6 233.2 232.9 235.6 235.7 237.7 239.1
l
Year-on-Year In(cid:29)ation (%)
All Consumer Prices 23.8 23.5 23.1 22.4 21.2 18.4 13.7 12.1 11.5 9.4 8.0 6.3 5.4
... Food Prices 27.8 2b8.3 28.1 26.5 25.0 22.8 16.3 15.1 14.8 10.8 9.5 6.6 4.9
... Non-Food Prices 20.3 19.2 18.8 18.7 17.9 14.4 11.4 9.5 8.7 8.2 6.8 6.1 5.8
Monthly In(cid:29)ation (%)
All Consumer Prices 1.8 1.7 1.3 0.2 0.8 0.7 1.2 0.7 1.3 0.9 0.4 0.9 0.9
− − −
... Food Prices 2.8 2.0 1.8 0.2 0.9 0.9 0.5 0.6 2.5 0.6 0.9 1.1 1.1
− − − −
... N

In [11]:
import pdfplumber

with pdfplumber.open('../data/bog_summary_jan2026.pdf') as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()
        if text and 'Treasury' in text and 'Bill' in text:
            print(f"\n=== Page {i} ===")
            print(text[:3000])
            print("\n" + "="*50)


=== Page 4 ===
27-Jan-2026 18:02:56 Summary of Macroeconomic and Financial Data c3
3a. Interest Rates (Percent Per Annum)
2024:12 2025:01 2025:02 2025:03 2025:04 2025:05 2025:06 2025:07 2025:08 2025:09 2025:10 2025:11 2025:12
Monetary Policy Rate 27.00 27.00 27.00 28.00 28.0i0 28.00 28.00 25.00 25.00 21.50 21.50 18.00 18.00
Interbank Weighted Average 27.03 27.06 27.04 26.28 26.92 27.02 27.02 22.76 23.28 23.14 21.00 20.51 16.29
Treasury Instruments
l
91-Day Bill (interest equivalent) 27.73 28.37 26.93 17.15 15.47 15.11 14.74 13.44 10.26 10.45 10.63 10.98 11.08
182-Day Bill (interest equivalent) 28.43 28.98 27.69 18.48 16.23 15.68 15.34 14.47 12.31 12.40 12.57 12.61 12.50
364-Day Bill (interest equivalent) 29.95 30.26 28.90 19.85 18.62 16.64 15.76 14.84 13.11 12.94 12.90 13.03 12.92
b
Deposit Rates
Demand Deposits 2.63 2.63 2.63 2.63 2.63 2.63 2.63 2.63 2.63 1.13 1.13 1.13 1.13
Savings Deposits 5.00 5.00 5.00 5.00 5.00 5.00 5.00 5.00 5.00 5.00 5.00 5.00 5.00
Time Deposits
... 3-months 1

In [12]:
import pandas as pd

# Data from Page 4 of the Jan 2026 PDF
# Covers Dec 2024 - Dec 2025
data_2025 = {
    'date': pd.date_range('2024-12-01', '2025-12-01', freq='MS'),
    'tbill_91': [27.73, 28.37, 26.93, 17.15, 15.47, 15.11, 14.74, 13.44, 10.26, 10.45, 10.63, 10.98, 11.08],
    'tbill_182': [28.43, 28.98, 27.69, 18.48, 16.23, 15.68, 15.34, 14.47, 12.31, 12.40, 12.57, 12.61, 12.50],
    'tbill_364': [29.95, 30.26, 28.90, 19.85, 18.62, 16.64, 15.76, 14.84, 13.11, 12.94, 12.90, 13.03, 12.92],
    'policy_rate': [27.00, 27.00, 27.00, 28.00, 28.00, 28.00, 28.00, 25.00, 25.00, 21.50, 21.50, 18.00, 18.00],
    'interbank_rate': [27.03, 27.06, 27.04, 26.28, 26.92, 27.02, 27.02, 22.76, 23.28, 23.14, 21.00, 20.51, 16.29],
    'avg_lending_rate': [30.25, 30.07, 30.12, 29.18, 27.40, 26.90, 27.00, 26.59, 24.15, 22.71, 22.22, 21.71, 20.45]
}

df_2025 = pd.DataFrame(data_2025)
print(df_2025)
print(f"\nShape: {df_2025.shape}")

         date  tbill_91  tbill_182  tbill_364  policy_rate  interbank_rate  \
0  2024-12-01     27.73      28.43      29.95         27.0           27.03   
1  2025-01-01     28.37      28.98      30.26         27.0           27.06   
2  2025-02-01     26.93      27.69      28.90         27.0           27.04   
3  2025-03-01     17.15      18.48      19.85         28.0           26.28   
4  2025-04-01     15.47      16.23      18.62         28.0           26.92   
5  2025-05-01     15.11      15.68      16.64         28.0           27.02   
6  2025-06-01     14.74      15.34      15.76         28.0           27.02   
7  2025-07-01     13.44      14.47      14.84         25.0           22.76   
8  2025-08-01     10.26      12.31      13.11         25.0           23.28   
9  2025-09-01     10.45      12.40      12.94         21.5           23.14   
10 2025-10-01     10.63      12.57      12.90         21.5           21.00   
11 2025-11-01     10.98      12.61      13.03         18.0      

In [13]:
import requests
import warnings
warnings.filterwarnings('ignore')

# BoG PDF URLs follow a pattern - download multiple years
pdfs_to_download = {
    'jan_2025': 'https://www.bog.gov.gh/wp-content/uploads/2025/01/Summary-of-Economic-and-Financial-Data-January-2025.pdf',
    'jan_2024': 'https://www.bog.gov.gh/wp-content/uploads/2024/01/Summary-of-Economic-and-Financial-Data-January-2024.pdf',
    'jan_2023': 'https://www.bog.gov.gh/wp-content/uploads/2023/01/Summary-of-Economic-and-Financial-Data-January-2023.pdf',
    'jan_2022': 'https://www.bog.gov.gh/wp-content/uploads/2022/01/Summary-of-Economic-and-Financial-Data-January-2022-1.pdf',
    'jan_2021': 'https://www.bog.gov.gh/wp-content/uploads/2021/01/Summary-of-Economic-Financial-Data-January-2021.pdf',
}

for name, url in pdfs_to_download.items():
    try:
        response = requests.get(url, verify=False, timeout=30)
        if response.status_code == 200:
            filepath = f'../data/bog_summary_{name}.pdf'
            with open(filepath, 'wb') as f:
                f.write(response.content)
            print(f"Downloaded: {name} ({len(response.content)} bytes)")
        else:
            print(f"Failed: {name} (Status {response.status_code})")
    except Exception as e:
        print(f"Error: {name} - {e}")

Downloaded: jan_2025 (405171 bytes)
Downloaded: jan_2024 (1124847 bytes)
Failed: jan_2023 (Status 404)
Downloaded: jan_2022 (853511 bytes)
Downloaded: jan_2021 (185251 bytes)


In [14]:
# Try alternative URLs for Jan 2023
alt_urls = [
    'https://www.bog.gov.gh/wp-content/uploads/2023/01/Summary-of-Economic-Financial-Data-January-2023.pdf',
    'https://www.bog.gov.gh/wp-content/uploads/2023/02/Summary-of-Economic-and-Financial-Data-January-2023.pdf',
    'https://www.bog.gov.gh/wp-content/uploads/2023/01/Summary-of-Economic-and-Financial-Data-January-2023-1.pdf',
]

for url in alt_urls:
    try:
        response = requests.get(url, verify=False, timeout=30)
        if response.status_code == 200:
            with open('../data/bog_summary_jan_2023.pdf', 'wb') as f:
                f.write(response.content)
            print(f"Success! {url}")
            print(f"Size: {len(response.content)} bytes")
            break
        else:
            print(f"Failed ({response.status_code}): {url}")
    except Exception as e:
        print(f"Error: {e}")

Success! https://www.bog.gov.gh/wp-content/uploads/2023/01/Summary-of-Economic-Financial-Data-January-2023.pdf
Size: 394730 bytes


In [15]:
import pdfplumber

pdfs = {
    'jan_2025': '../data/bog_summary_jan_2025.pdf',
    'jan_2024': '../data/bog_summary_jan_2024.pdf',
    'jan_2022': '../data/bog_summary_jan_2022.pdf',
    'jan_2021': '../data/bog_summary_jan_2021.pdf',
}

for name, path in pdfs.items():
    print(f"\n{'='*60}")
    print(f"=== {name} ===")
    print(f"{'='*60}")
    try:
        with pdfplumber.open(path) as pdf:
            for i, page in enumerate(pdf.pages):
                text = page.extract_text()
                if text and '91-Day' in text and 'Bill' in text:
                    print(f"Page {i}:")
                    print(text[:2500])
                    break
    except Exception as e:
        print(f"Error: {e}")


=== jan_2025 ===
Page 4:
24-Jan-2025 19:24:45 Summary of Macroeconomic and Financial Data c3
3a. Interest Rates
2023:12 2024:01 2024:03 2024:04 2024:06 2024:07 2024:08 2024:09 2024:10 2024:11 2024:12
Monetary Policy Rate % 30.00 29.00 29.00 i29.00 29.00 29.00 29.00 27.00 27.00 27.00 27.00
Interbank Weighted Average % p.a. 30.19 29.93 28.48 28.68 28.80 28.83 28.84 28.84 27.69 27.03 27.03
Treasury Instruments (interest equivalent)
l
91-Day Bill % p.a. 29.39 28.93 26.40 25.68 24.91 24.80 24.82 25.07 25.80 26.89 27.73
182-Day Bill % p.a. 31.70 31.44 28.90 28.03 26.84 26.75 26.74 26.82 27.01 27.76 28.43
364-Day Bill % p.a. 32.97 32.05 29.50 28.64 27.83 27.81 27.84 28.13 28.70 29.30 29.95
b
Deposit Rates
Demand Deposits % p.a. 2.63 2.63 2.63 2.63 2.63 2.63 2.63 2.63 2.63 2.63 2.63
Savings Deposits % p.a. 5.00 5.00 5.00 5.00 5.00 5.00 5.00 5.00 5.00 5.00 5.00
Time Deposits
... 3-months % p.a. 10.50 10.50 10.50 10.50 10.50 10.50 10.50 10.50 10.50 10.50 10.50
... 6-months u% p.a. 11.00 11.00 1

In [16]:
# Extract Jan 2023 PDF
with pdfplumber.open('../data/bog_summary_jan_2023.pdf') as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()
        if text and '91-Day' in text and 'Bill' in text:
            print(f"Page {i}:")
            print(text[:2500])
            break

Page 4:
C
27-Jan-2023 18:07:48 Summary of Macroeconomic and Financial Data 3
3. Interest Rates
2021:12 2022:01 2022:02 2022:03 2022:06 2022:07 2022:08 2022:09 2022:10 2022:11 2022:12
Money Market
I
Monetary Policy Rate % 14.50 14.50 14.50 17.00 19.00 19.00 22.00 22.00 24.50 27.00 27.00
Interbank Weighted Average % p.a. 12.68 12.81 12.96 13.32 19.92 21.25 21.93 22.05 23.98 25.80 25.51
Treasury Instruments (interest equivalent)
91-Day Bill % p.a. 12.49 12.55 12.82 13.49 24.15 26.16 27.68 29.65 31.53 34.62 35.48
182-Day Bill % p.a. 13.19 13L.21 13.27 13.76 25.55 27.38 29.12 30.94 32.61 35.68 36.23
364-Day Bill % p.a. 16.46 16.70 16.97 17.01 27.14 27.67 28.92 30.24 32.32 35.26 36.06
Secondary Market
2-Year Note % p.a. 20.78 20.91 19.58 20.23 30.18 30.62 35.70 39.35 43.76 42.87 59.49
3-Year Bond % p.a. 11.56 12.19 18.49 15.03 22.86 25.68 32.39 41.68 67.06 43.71 43.69
5-Year Bond % p.a. 21.01 21.40 20.64 20.84 30.63 30.01 36.35 39.73 44.81 45.19 43.68
B
6-Year Bond % p.a. 20.81 21.19 20.68 2

In [17]:
import pandas as pd

# Compile all T-bill data from the PDFs
# Format: (date, 91-day, 182-day, 364-day, policy_rate)

records = [
    # From Jan 2021 PDF (Dec 2019 - Dec 2020)
    ('2019-12-01', 14.69, 15.15, 17.88, 16.00),
    ('2020-03-01', 14.73, 15.17, 17.74, 14.50),
    ('2020-04-01', 14.05, 14.27, 16.76, 14.50),
    ('2020-05-01', 13.95, 14.02, 16.73, 14.50),
    ('2020-06-01', 13.97, 14.05, 16.87, 14.50),
    ('2020-07-01', 13.95, 14.05, 16.87, 14.50),
    ('2020-08-01', 14.02, 14.09, 16.89, 14.50),
    ('2020-09-01', 14.02, 14.12, 16.95, 14.50),
    ('2020-10-01', 14.05, 14.11, 16.99, 14.50),
    ('2020-11-01', 14.05, 14.12, 16.97, 14.50),
    ('2020-12-01', 14.08, 14.13, 16.98, 14.50),
    
    # From Jan 2022 PDF (Jan 2021 - Dec 2021)
    ('2021-01-01', 14.09, 14.14, 16.96, 14.50),
    ('2021-03-01', 13.02, 13.78, 16.70, 14.50),
    ('2021-06-01', 12.65, 13.40, 16.34, 13.50),
    ('2021-07-01', 12.56, 13.37, 16.36, 13.50),
    ('2021-08-01', 12.49, 13.27, 16.20, 13.50),
    ('2021-09-01', 12.47, 13.20, 16.12, 13.50),
    ('2021-10-01', 12.46, 13.16, 16.24, 13.50),
    ('2021-11-01', 12.48, 13.17, 16.28, 14.50),
    ('2021-12-01', 12.49, 13.19, 16.46, 14.50),
    
    # From Jan 2024 PDF (Dec 2022 - Dec 2023)
    ('2022-12-01', 35.48, 36.23, 36.06, 27.00),
    ('2023-01-01', 35.62, 35.84, 35.76, 28.00),
    ('2023-02-01', 35.67, 35.73, 34.92, 28.00),
    ('2023-03-01', 20.38, 23.01, 26.67, 29.50),
    ('2023-04-01', 19.67, 22.29, 27.04, 29.50),
    ('2023-06-01', 21.77, 24.58, 28.66, 29.50),
    ('2023-07-01', 24.64, 26.44, 30.00, 30.00),
    ('2023-08-01', 26.35, 27.84, 30.88, 30.00),
    ('2023-09-01', 28.20, 29.84, 32.29, 30.00),
    ('2023-10-01', 29.40, 31.37, 33.16, 30.00),
    ('2023-11-01', 29.72, 31.88, 33.45, 30.00),
    ('2023-12-01', 29.39, 31.70, 32.97, 30.00),
    
    # From Jan 2025 PDF (Jan 2024 - Dec 2024)
    ('2024-01-01', 28.93, 31.44, 32.05, 29.00),
    ('2024-03-01', 26.40, 28.90, 29.50, 29.00),
    ('2024-04-01', 25.68, 28.03, 28.64, 29.00),
    ('2024-06-01', 24.91, 26.84, 27.83, 29.00),
    ('2024-07-01', 24.80, 26.75, 27.81, 29.00),
    ('2024-08-01', 24.82, 26.74, 27.84, 29.00),
    ('2024-09-01', 25.07, 26.82, 28.13, 27.00),
    ('2024-10-01', 25.80, 27.01, 28.70, 27.00),
    ('2024-11-01', 26.89, 27.76, 29.30, 27.00),
    ('2024-12-01', 27.73, 28.43, 29.95, 27.00),
    
    # From Jan 2026 PDF (Jan 2025 - Dec 2025)
    ('2025-01-01', 28.37, 28.98, 30.26, 27.00),
    ('2025-02-01', 26.93, 27.69, 28.90, 27.00),
    ('2025-03-01', 17.15, 18.48, 19.85, 28.00),
    ('2025-04-01', 15.47, 16.23, 18.62, 28.00),
    ('2025-05-01', 15.11, 15.68, 16.64, 28.00),
    ('2025-06-01', 14.74, 15.34, 15.76, 28.00),
    ('2025-07-01', 13.44, 14.47, 14.84, 25.00),
    ('2025-08-01', 10.26, 12.31, 13.11, 25.00),
    ('2025-09-01', 10.45, 12.40, 12.94, 21.50),
    ('2025-10-01', 10.63, 12.57, 12.90, 21.50),
    ('2025-11-01', 10.98, 12.61, 13.03, 18.00),
    ('2025-12-01', 11.08, 12.50, 12.92, 18.00),
]

# Create DataFrame
df = pd.DataFrame(records, columns=['date', 'tbill_91', 'tbill_182', 'tbill_364', 'policy_rate'])
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').drop_duplicates(subset='date').reset_index(drop=True)

print(f"Total observations: {len(df)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nLast 5 rows:")
print(df.tail())

# Save to CSV
df.to_csv('../data/ghana_tbill_rates.csv', index=False)
print("\nSaved to data/ghana_tbill_rates.csv")

Total observations: 54
Date range: 2019-12-01 00:00:00 to 2025-12-01 00:00:00

First 5 rows:
        date  tbill_91  tbill_182  tbill_364  policy_rate
0 2019-12-01     14.69      15.15      17.88         16.0
1 2020-03-01     14.73      15.17      17.74         14.5
2 2020-04-01     14.05      14.27      16.76         14.5
3 2020-05-01     13.95      14.02      16.73         14.5
4 2020-06-01     13.97      14.05      16.87         14.5

Last 5 rows:
         date  tbill_91  tbill_182  tbill_364  policy_rate
49 2025-08-01     10.26      12.31      13.11         25.0
50 2025-09-01     10.45      12.40      12.94         21.5
51 2025-10-01     10.63      12.57      12.90         21.5
52 2025-11-01     10.98      12.61      13.03         18.0
53 2025-12-01     11.08      12.50      12.92         18.0

Saved to data/ghana_tbill_rates.csv


In [18]:
fx_raw = pd.read_csv('../data/usd_ghs.csv', skiprows=3)
print("Columns:", fx_raw.columns.tolist())
print(fx_raw.head())

Columns: ['2015-01-05', '3.180000066757202']
   2015-01-05  3.180000066757202
0  2015-01-06               3.18
1  2015-01-07               3.19
2  2015-01-08               3.19
3  2015-01-09               3.20
4  2015-01-12               3.20
